# Feature Engineering & RFM Analysis

In [29]:
import pandas as pd
import numpy as np

In [30]:
df = pd.read_csv('../data/processed/cleaned_transactions.csv')

In [31]:
df.shape


(404876, 10)

In [32]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404876 entries, 0 to 404875
Data columns (total 10 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   index        404876 non-null  int64  
 1   InvoiceNo    404876 non-null  object 
 2   StockCode    404876 non-null  object 
 3   Description  404876 non-null  object 
 4   Quantity     404876 non-null  int64  
 5   InvoiceDate  404876 non-null  object 
 6   UnitPrice    404876 non-null  float64
 7   CustomerID   404876 non-null  float64
 8   Country      404876 non-null  object 
 9   TotalAmount  404876 non-null  float64
dtypes: float64(3), int64(2), object(5)
memory usage: 30.9+ MB


In [33]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

In [34]:
df.dtypes

index                   int64
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
TotalAmount           float64
dtype: object

In [35]:
last_date = df['InvoiceDate'].max()
last_date

Timestamp('2011-12-09 12:50:00')

In [36]:
Reference_date=last_date+(pd.Timedelta(days=1))
Reference_date

Timestamp('2011-12-10 12:50:00')

In [37]:
df.groupby('CustomerID')['InvoiceDate'].max()

CustomerID
12346.0   2011-01-18 10:17:00
12347.0   2011-12-07 15:52:00
12348.0   2011-09-25 13:13:00
12349.0   2011-11-21 09:51:00
12350.0   2011-02-02 16:01:00
                  ...        
18280.0   2011-03-07 09:52:00
18281.0   2011-06-12 10:53:00
18282.0   2011-12-02 11:43:00
18283.0   2011-12-06 12:02:00
18287.0   2011-10-28 09:29:00
Name: InvoiceDate, Length: 4362, dtype: datetime64[ns]

In [38]:
last_purchase = df.groupby('CustomerID')['InvoiceDate'].max()

In [39]:
recency = Reference_date - last_purchase

In [40]:
recency = recency.dt.days

In [41]:
recency

CustomerID
12346.0    326
12347.0      2
12348.0     75
12349.0     19
12350.0    310
          ... 
18280.0    278
18281.0    181
18282.0      8
18283.0      4
18287.0     43
Name: InvoiceDate, Length: 4362, dtype: int64

In [42]:
frequency = df.groupby('CustomerID')['InvoiceNo'].nunique()

In [43]:
frequency

CustomerID
12346.0     2
12347.0     7
12348.0     4
12349.0     1
12350.0     1
           ..
18280.0     1
18281.0     1
18282.0     3
18283.0    16
18287.0     3
Name: InvoiceNo, Length: 4362, dtype: int64

In [44]:
monetary = df.groupby('CustomerID')['TotalAmount'].sum()

In [45]:
monetary

CustomerID
12346.0       0.00
12347.0    4310.00
12348.0    1437.24
12349.0    1457.55
12350.0     294.40
            ...   
18280.0     180.60
18281.0      80.82
18282.0     176.60
18283.0    2088.93
18287.0    1837.28
Name: TotalAmount, Length: 4362, dtype: float64

In [46]:
rfm = pd.DataFrame({
    'Recency': recency,
    'Frequency': frequency,
    'Monetary': monetary
})

In [47]:
rfm.head()
rfm.shape

(4362, 3)

In [48]:
rfm.to_csv('../data/processed/rfm_features.csv')

## RFM Feature Engineering

Grouped transaction-level data by CustomerID to build customer-level features:

- **Recency**: days since each customer's last purchase, measured against a reference date (one day after the dataset's last transaction: 2011-12-10)
- **Frequency**: count of distinct invoices per customer
- **Monetary**: total amount spent per customer

Result: 4,362 customers with Recency, Frequency, and Monetary columns — saved to `data/processed/rfm_features.csv` for use in CLV calculation.